# Mixed-fall and hybrid physical-event runs

The lag-1 mixed-fall benchmark and later hybrid/context grids are kept in different output directories. Every long cell always passes `--resume`.

<!-- reviewer-resume-contract -->
## Execution and resume contract

This notebook is aligned with the reviewer-revision implementation. Expensive work is checkpointed and safe to restart with the same configuration. Do not change methods, seeds, thresholds, or output paths while resuming. Saved outputs remain provisional until the compute-machine run and verification gates complete.


In [ ]:
from pathlib import Path
import os, shlex, subprocess, sys
def find_package_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for base in (current, *current.parents):
        for candidate in (base, base / 'calcium-transient-rising-flank'):
            if (candidate / 'pyproject.toml').is_file() and (candidate / 'examples' / 'dynamic_extensions.py').is_file():
                return candidate
    raise FileNotFoundError('Could not locate the calcium-transient-rising-flank package root')

PACKAGE_ROOT = find_package_root()
RUNNER_PYTHON = next((str(path) for path in (PACKAGE_ROOT / '.venv/bin/python', PACKAGE_ROOT / '.venv/Scripts/python.exe') if path.is_file()), sys.executable)
RUNNER_ENV = os.environ.copy()
RUNNER_ENV['MPLBACKEND'] = 'Agg'
RUNNER_ENV['PYTHONUNBUFFERED'] = '1'
SOURCE_ROOT = str(PACKAGE_ROOT / 'src')
RUNNER_ENV['PYTHONPATH'] = os.pathsep.join(filter(None, (SOURCE_ROOT, RUNNER_ENV.get('PYTHONPATH'))))
sys.path.insert(0, SOURCE_ROOT)
from calcium_transient_rising_flank.checkpointing import format_progress
RUN_MIXED_FALL = False
RUN_HYBRID = False
N_SEEDS = 8
N_SURROGATES = 1000
OUTPUT_ROOT = PACKAGE_ROOT / 'outputs/revision_campaign'
COMMON = [
    RUNNER_PYTHON, 'examples/dynamic_extensions.py',
    '--methods', 'cgc,cgc-star', '--n-seeds', str(N_SEEDS),
    '--n-surrogates', str(N_SURROGATES), '--resume',
]
def launch(command, enabled):
    print(f'[notebook] command: {shlex.join(command)}', flush=True)
    if enabled:
        print(format_progress(0, 1, label='Notebook stage') + ' | running', flush=True)
        subprocess.run(command, cwd=PACKAGE_ROOT, env=RUNNER_ENV, check=True)
        print(format_progress(1, 1, label='Notebook stage') + ' | complete', flush=True)
    else:
        print('[notebook] preview only; enable the RUN_* toggle to execute', flush=True)

In [ ]:
mixed_fall_command = [
    *COMMON, '--grid', 'lag1_context1',
    '--output-dir', str(OUTPUT_ROOT / 'mixed_fall'),
]
launch(mixed_fall_command, RUN_MIXED_FALL)

In [ ]:
hybrid_command = [
    *COMMON, '--grid', 'lag2_context2,bout_bounded_lag3_context4',
    '--output-dir', str(OUTPUT_ROOT / 'hybrid_event'),
]
launch(hybrid_command, RUN_HYBRID)